# Task 3: Outer Vector Add

Bot task id: `03_outer_vector_add`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/03_outer_vector_add.ipynb)

## Условие

Вычислите *внешнюю сумму* двух векторов `x` и `y`, то есть матрицу `z` размера `(N1, N0)`:

$$z[i, j] = x[j] + y[i], \quad i = 0 \dots N_1-1,\ j = 0 \dots N_0-1$$

**Требования**
- сетка из одной программы, размеры блоков равны размерам векторов (`B0 == N0`, `B1 == N1`);
- маска не требуется (нет неполных блоков);
- использовать `tl.arange`, `tl.load`, `tl.store`;
- решение должно работать на GPU (CUDA).

**Подсказка**
- `row_ids = tl.arange(0, N1)`, `col_ids = tl.arange(0, N0)`;
- результат: `y[:, None] + x[None, :]`;
- смещения для записи: `offs = row_ids[:, None] * N0 + col_ids[None, :]`, затем `tl.store(z_ptr + offs, z)`.

Обёртку `outer_vector_add_triton` пишете сами. Проверка: `N0 = 32`, `N1 = 16`.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `03_outer_vector_add.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 03_outer_vector_add.py
import torch
import triton
import triton.language as tl


@triton.jit
def _outer_add_kernel(x_ptr, y_ptr, z_ptr, N0: tl.constexpr, N1: tl.constexpr):
    # YOUR CODE HERE
    pass


def outer_vector_add_triton(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    # YOUR CODE HERE
    raise NotImplementedError

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "03_outer_vector_add.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

x, y = torch.randn(32, device="cuda"), torch.randn(16, device="cuda")
torch.testing.assert_close(sol.outer_vector_add_triton(x, y), y[:, None] + x[None, :])

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("03_outer_vector_add.py")